# 03 — The state is just bytes

There is no sentencepiece download. Each UTF-8 byte is a token id in `0..255`. Id `256` is padding and is not a byte, so it never collides with text.

A personal chat log is a few million of these bytes. That is the whole corpus size this model is built for. A string longer than `max_len` keeps its **last** bytes, so a packed thread still ends on the newest line.

**You implement:** `clean_state`, `encode_state`.



In [ ]:
from __future__ import annotations


def clean_state(text: str) -> str | None:
    """Trim, collapse whitespace. None if empty or a bare URL."""
    raise NotImplementedError("clean_state")


def encode_state(text: str, max_len: int, pad_id: int = 256) -> tuple[list[int], list[float]]:
    """Return (ids length max_len, mask of 1.0 on real bytes)."""
    raise NotImplementedError("encode_state")



<details>
<summary><b>Solution</b> (try yourself first — click to expand)</summary>

```python
def clean_state(text: str) -> str | None:
    text = " ".join(text.split())
    if not text:
        return None
    lower = text.lower()
    if (lower.startswith("http://") or lower.startswith("https://")) and " " not in text:
        return None
    return text

def encode_state(text: str, max_len: int, pad_id: int = 256) -> tuple[list[int], list[float]]:
    ids = list(text.encode("utf-8"))
    if len(ids) > max_len:
        ids = ids[-max_len:]
    if not ids:
        ids = [ord(" ")]
    width = len(ids)
    pad = max_len - width
    return ids + [pad_id] * pad, [1.0] * width + [0.0] * pad
```

Copy into the stub cell above, then run **Check**.

</details>



## Check



In [ ]:
assert clean_state("  hey   there ") == "hey there"
assert clean_state("https://example.com/a") is None
ids, mask = encode_state("hi", max_len=4)
assert ids == [ord("h"), ord("i"), 256, 256]
assert mask == [1.0, 1.0, 0.0, 0.0]
tail, _ = encode_state("abcdef", max_len=3)
assert tail == [ord("d"), ord("e"), ord("f")]
print(ids, mask)

